<a href="https://colab.research.google.com/github/koki-shiroyama0430/Complete-Data-Science-Bootcamp---Udemy/blob/main/03_MNIST_Handwritten_Digit_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries

In [1]:
import numpy as np
import tensorflow as tf

import tensorflow_datasets as tfds

### Loading Dataset



In [2]:
# Load MNIST: with_info for metadata, as_supervised for (input, label) tuples
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

# Split into train and test sets
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

# Define number of validation samples (10% of training data)
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

# Define number of test samples
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

# Scale pixel values to [0,1] range for faster convergence
def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255.
    return image, label

scaled_train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

# Shuffle and split into Train and Validation
BUFFER_SIZE = 10000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

# Set batch size for mini-batch gradient descent
BATCH_SIZE = 100

# Batching datasets to update weights every 100 samples
train_data = train_data.batch(BATCH_SIZE)

# Batching validation/test data as a single block for faster evaluation
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

# Extract one batch from the validation pipeline for evaluation
validation_inputs, validation_targets = next(iter(validation_data))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.TLC3MI_3.0.1/mnist-train.tfrecord*...:   0%|          | 0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.TLC3MI_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


### Model Configuration
Define the basic dimensions of neural network.
Use a Sequential model to stack layers linearly.

In [3]:
# Hyperparameters: Defining network dimensions
input_size = 784     # 28x28 pixels
output_size = 10    # Digits 0-9
hidden_layer_size = 50 # Number of neurons in each hidden layer

# Model Architecture: Defining the feed-forward structure
model = tf.keras.Sequential([
    # Convert 2D image (28x28) into 1D vector (784)
    tf.keras.layers.Flatten(input_shape=(28,28,1)),

    # Hidden layers with ReLU to capture non-linear patterns
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),

    # Output layer with Softmax to output class probabilities
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# Display the model's architecture and total trainable parameters
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50)             │        39,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,310 (165.27 KB)

 Trainable params: 42,310 (165.27 KB)

 Non-trainable params: 0 (0.00 B)

### Compile
Define how the model should learn.

In [4]:
# Configure the learning process
model.compile(
    optimizer='adam',  # Adaptive moment Estimation
    loss='sparse_categorical_crossentropy',  # Loss function for multi-class classification
    metrics=['accuracy']  # Monitor the percentage of correct predictions
)

### Model Training

In [5]:
# Set the number of training iterations
NUM_EPOCHS = 5

# Start the training process
# The model learns from train_data while monitoring performance on validation_data
model.fit(
    train_data,
    epochs=NUM_EPOCHS,
    validation_data=(validation_inputs, validation_targets),
    verbose=2
)

Epoch 1/5
540/540 - 11s - 21ms/step - accuracy: 0.8811 - loss: 0.4261 - val_accuracy: 0.9372 - val_loss: 0.2155
Epoch 2/5
540/540 - 8s - 16ms/step - accuracy: 0.9453 - loss: 0.1887 - val_accuracy: 0.9492 - val_loss: 0.1698
Epoch 3/5
540/540 - 8s - 15ms/step - accuracy: 0.9577 - loss: 0.1447 - val_accuracy: 0.9575 - val_loss: 0.1364
Epoch 4/5
540/540 - 5s - 10ms/step - accuracy: 0.9645 - loss: 0.1185 - val_accuracy: 0.9610 - val_loss: 0.1240
Epoch 5/5
540/540 - 5s - 9ms/step - accuracy: 0.9706 - loss: 0.0988 - val_accuracy: 0.9658 - val_loss: 0.1024


### Model Evaluation

In [6]:
# Evaluate the final model performance using the test dataset
test_loss, test_accuracy = model.evaluate(test_data)

# Print the final results in a clean format
print('Test loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9680 - loss: 0.1085
Test loss: 0.11. Test accuracy: 96.80%


In [9]:
# Save the trained model in Keras format
# This 'keras' file is what you will later upload to Vertex AI
model.save('mnist_model.keras')